In [ ]:
import os
import re
import time
import random
import pandas as pd
from tqdm import tqdm
from openai import OpenAI, APITimeoutError, RateLimitError, APIError

MODEL_ID = "moonshotai/kimi-k2.5"
RESULTS_PATH = "../results/importance.csv"

MAX_TOKENS = 64
REQUEST_TIMEOUT = 30
MAX_RETRIES = 3

CALL_INTERVAL_S = 0.8
RATE_LIMIT_BASE_SLEEP_S = 8
JITTER_S = 0.4
QUESTION_SLEEP_S = 0.8

QUESTION_SEED = 2026
TOTAL_QUESTIONS = 80
QUESTIONS_PER_RUN = 10
STEP_LIMIT = 4
TARGET_VALID_QUESTIONS = 40
RESET_EXISTING_RESULTS = False

NVIDIA_API_KEY = (os.environ.get("NVIDIA_API_KEY") or "").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Please set NVIDIA_API_KEY in your environment.")
if any(ord(c) > 127 for c in NVIDIA_API_KEY):
    raise ValueError("NVIDIA_API_KEY contains non-ASCII characters. Re-export it with plain ASCII text.")
if not NVIDIA_API_KEY.startswith("nvapi-"):
    raise ValueError("NVIDIA_API_KEY should start with 'nvapi-'.")

client = OpenAI(
    api_key=NVIDIA_API_KEY,
    base_url="https://integrate.api.nvidia.com/v1",
)

LAST_CALL_TS = 0.0
NUM_PATTERN = re.compile(r"-?\d+(?:\.\d+)?")


In [ ]:
def generate_questions(n=50, seed=QUESTION_SEED):
    rng = random.Random(seed)
    questions = []
    for _ in range(n):
        a = rng.randint(2, 20)
        b = rng.randint(2, 20)
        c = rng.randint(1, 10)
        q = f"{a} * {b} + {c} = ?"
        questions.append(q)
    return questions

questions = generate_questions(TOTAL_QUESTIONS)
questions[:5]


In [ ]:
def normalize_number_str(text):
    s = str(text).strip()
    if not s:
        return None
    try:
        x = float(s)
    except ValueError:
        return s
    if x.is_integer():
        return str(int(x))
    return f"{x:.10g}"


def extract_last_number(text):
    if text is None:
        return None
    matches = NUM_PATTERN.findall(str(text).replace(",", ""))
    if not matches:
        return None
    return normalize_number_str(matches[-1])


def extract_text(res):
    msg = res.choices[0].message
    content = getattr(msg, "content", None)

    if isinstance(content, str) and content.strip():
        return content.strip()

    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                text = item.get("text")
            else:
                text = getattr(item, "text", None)
            if text:
                parts.append(text)
        if parts:
            return "\n".join(parts).strip()

    reasoning = getattr(msg, "reasoning", None)
    if isinstance(reasoning, str) and reasoning.strip():
        return reasoning.strip()

    return None


def _throttle():
    global LAST_CALL_TS
    wait_s = CALL_INTERVAL_S - (time.time() - LAST_CALL_TS)
    if wait_s > 0:
        time.sleep(wait_s)


def call_llm(prompt, max_tokens=MAX_TOKENS, timeout=REQUEST_TIMEOUT, retries=MAX_RETRIES):
    global LAST_CALL_TS
    for i in range(retries):
        try:
            _throttle()
            res = client.chat.completions.create(
                model=MODEL_ID,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=max_tokens,
                temperature=0,
                timeout=timeout,
            )
            LAST_CALL_TS = time.time()
            text = extract_text(res)
            if text:
                return text

            if i < retries - 1:
                wait_s = 1 + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] empty response text, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
        except RateLimitError:
            LAST_CALL_TS = time.time()
            if i < retries - 1:
                wait_s = RATE_LIMIT_BASE_SLEEP_S * (2 ** i) + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] RateLimitError, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
            else:
                print("[fail] RateLimitError", flush=True)
        except (APITimeoutError, APIError) as e:
            LAST_CALL_TS = time.time()
            if i < retries - 1:
                wait_s = (2 ** i) + random.uniform(0, JITTER_S)
                print(f"[retry {i+1}/{retries}] {type(e).__name__}, sleep {wait_s:.1f}s", flush=True)
                time.sleep(wait_s)
            else:
                print(f"[fail] {type(e).__name__}", flush=True)

    return None


def generate_cot(question):
    prompt = f"""
Solve step by step:

{question}
"""
    return call_llm(prompt, max_tokens=128)


def get_answer(reasoning):
    prompt = f"""
Given reasoning:

{reasoning}

Return ONLY the final numeric answer.
Rules:
- Output a single number only.
- No words or explanation.
"""
    raw = call_llm(prompt, max_tokens=16)
    return extract_last_number(raw)


In [ ]:
if RESET_EXISTING_RESULTS and os.path.exists(RESULTS_PATH):
    os.remove(RESULTS_PATH)

if os.path.exists(RESULTS_PATH):
    old_df = pd.read_csv(RESULTS_PATH)
    old_df = old_df[["question", "step_index", "changed"]] if not old_df.empty else old_df
    records = old_df.to_dict("records")
else:
    records = []

processed_questions = {r["question"] for r in records if isinstance(r.get("question"), str)}
pending_questions = [q for q in questions if q not in processed_questions]
pending_questions = pending_questions[:QUESTIONS_PER_RUN]

print(f"[resume] processed={len(processed_questions)}/{len(questions)}, run_now={len(pending_questions)}", flush=True)

for qi, q in enumerate(tqdm(pending_questions), start=1):
    print(f"[importance] question {qi}/{len(pending_questions)}: {q}", flush=True)

    cot = generate_cot(q)
    if not cot:
        tqdm.write(f"[skip] empty cot: {q}")
        continue

    lines = [l for l in cot.split("\n") if l.strip()]
    if not lines:
        tqdm.write(f"[skip] no reasoning lines: {q}")
        continue

    base = get_answer(cot)
    if not base:
        tqdm.write(f"[skip] empty base answer: {q}")
        continue

    for i in range(min(len(lines), STEP_LIMIT)):
        modified = "\n".join(lines[:i] + lines[i+1:])
        new = get_answer(modified)
        if not new:
            continue

        records.append({
            "question": q,
            "step_index": i,
            "changed": new != base,
        })

    pd.DataFrame(records, columns=["question", "step_index", "changed"]).to_csv(RESULTS_PATH, index=False)
    time.sleep(QUESTION_SLEEP_S)

df_imp = pd.DataFrame(records, columns=["question", "step_index", "changed"])
done_questions = df_imp["question"].nunique() if not df_imp.empty else 0
print(f"[summary] rows={len(df_imp)}, unique_questions={done_questions}", flush=True)
print(f"[target] progress={done_questions}/{TARGET_VALID_QUESTIONS}", flush=True)
df_imp.head()


In [ ]:
df_imp.to_csv("../results/importance.csv", index=False)
